<div style="font-size:2.3em;font-weight:600;margin-bottom:10px">Agentic Artificial Intelligence</div>
<div style="font-size:1.7em;margin-bottom:14px">Unit 07 · Tracing</div>

Today we learn to **record** what an agent does.

<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px 32px"><div><b>1</b>&nbsp; Opening: a flight recorder for agents</div><div><b>6</b>&nbsp; Recording an agent</div><div><b>2</b>&nbsp; Connecting to Langfuse</div><div><b>7</b>&nbsp; Finding the right recording: labels</div><div><b>3</b>&nbsp; The simplest recording: @observe</div><div><b>8</b>&nbsp; Reading traces, and good habits</div><div><b>4</b>&nbsp; Steps inside steps</div><div><b>9</b>&nbsp; Your exercises</div><div><b>5</b>&nbsp; A real model call, written by hand</div><div><b>10</b>&nbsp; Wrap-up</div></div>

<div style="height:44px"></div>

## Part 1 of 10 · Opening: a flight recorder for agents

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What is a trace, and why do we need one?**

#### 1.1 · A trace

A **trace** is the recording of one run: its steps, what went in, what came out, and how long it took.

#### 1.2 · Why do agents need this?

**Observability** means: you can see from the outside what happens inside.

**1.** The same input can give a different output.

**2.** One question can mean many model calls and tools.

**3.** Each model call costs money. Providers charge per **token**, the pieces of text that a model reads and writes.

**4.** How do you know that your agent is doing a good job?

#### 1.3 · Langfuse

Our code sends the recordings to **Langfuse**, and Langfuse shows them on a website.

<div style="height:44px"></div>

## Part 2 of 10 · Connecting to Langfuse

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How does our code reach Langfuse?**

#### 2.1 · What you need at home

A free account at Langfuse, and three lines in your `.env` file:

```
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

In [ ]:
# 2.2 · Are the keys there?
import os
from dotenv import load_dotenv

load_dotenv()

langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
langfuse_base_url = os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")

if langfuse_public_key and langfuse_secret_key:
    print("✅ Langfuse credentials found!")
    print(f"   Public Key: ...{langfuse_public_key[-4:]}")
    print(f"   Base URL: {langfuse_base_url}")
else:
    print("⚠️  Langfuse credentials not found!")
    print("   Please add to your .env file:")
    print("   LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("   LANGFUSE_SECRET_KEY=sk-lf-...")
    print("   LANGFUSE_BASE_URL=https://cloud.langfuse.com  # or your self-hosted URL")

In [ ]:
# 2.3 · The Langfuse client
from langfuse import get_client

langfuse = get_client()

print("Langfuse client initialized!")
print(f"Base URL: {langfuse.base_url if hasattr(langfuse, 'base_url') else 'default'}")

<div style="height:44px"></div>

## Part 3 of 10 · The simplest recording: @observe

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What is the simplest way to record a function?**

In [ ]:
# 3.1 · A normal function, with one new line
from langfuse import observe

@observe()
def calculate_total(items):
    total = sum(item['price'] * item['quantity'] for item in items)
    return {"total": total, "item_count": len(items)}

In [ ]:
# 3.2 · Call it, and send the record
result = calculate_total([
    {"name": "Apple", "price": 1.50, "quantity": 3},
    {"name": "Banana", "price": 0.75, "quantity": 5}
])

print(f"Result: {result}")

langfuse.flush()
print("\n✅ Trace sent to Langfuse! Check your dashboard. "
      "On the langfuse dashboard, you can see the trace in the traces tab.")

#### 3.3 · Let's look at the recording

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
calculate_total

input:   args, with our list: Apple, Banana
output:  total 8.25, item_count 2
time:    less than a second
```

</details>

<div style="height:44px"></div>

## Part 4 of 10 · Steps inside steps

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How do we record steps inside steps?**

In [ ]:
# 4.1 · A step with two steps inside
from langfuse import get_client

langfuse = get_client()

with langfuse.start_as_current_observation(
    as_type="span",
    name="process-order",
    input={"order_id": "12345", "customer": "Alice"}
) as root_span:

    with langfuse.start_as_current_observation(
        as_type="span",
        name="validate-order"
    ) as validation_span:
        validation_span.update(output={"status": "valid", "items": 3})

    with langfuse.start_as_current_observation(
        as_type="generation",
        name="generate-receipt",
        model="gemini-3.5-flash-lite",
        input={"order_id": "12345", "total": 7.25}
    ) as generation:
        receipt_text = "Thank you for your order! Total: $7.25"
        generation.update(
            output=receipt_text,
            usage_details={"input_tokens": 10, "output_tokens": 15}
        )

    root_span.update(output={"order_processed": True, "receipt": receipt_text})

In [ ]:
# 4.2 · Send it
langfuse.flush()
print("✅ Nested trace sent to Langfuse!")

#### 4.3 · Let's look at the tree

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
process-order
├─ validate-order
└─ generate-receipt   10 input tokens, 15 output tokens

time:  less than a second for each step
```

</details>

<div style="height:44px"></div>

## Part 5 of 10 · A real model call, written by hand

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What does Langfuse get from a real model call?**

In [ ]:
# 5.1 · The model and the client
from langchain.chat_models import init_chat_model
from langfuse import get_client

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")

langfuse = get_client()

In [ ]:
# 5.2 · One model call, recorded by hand
with langfuse.start_as_current_observation(
    as_type="generation",
    name="math-question",
    model="gemini-3.5-flash-lite",
    input={"messages": [{"role": "user", "content": "What is 15 * 7?"}]}
) as gen:
    response = llm.invoke("What is 15 * 7?")

    gen.update(
        output=response.text,
        usage_details={
            "input_tokens": response.response_metadata
                .get("usage_metadata", {}).get("input_token_count", 0),
            "output_tokens": response.response_metadata
                .get("usage_metadata", {}).get("output_token_count", 0)
        }
    )

    print(f"LLM Response: {response.text}")

langfuse.flush()
print("\n✅ LLM call traced in Langfuse!")

#### 5.3 · Let's look at the recording

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
math-question

input:   What is 15 * 7?
output:  15 * 7 = 105
tokens:  0 input, 0 output
time:    14 s
```

This time includes a pause of 4.5 seconds that only our test adds.

</details>

<div style="height:44px"></div>

## Part 6 of 10 · Recording an agent

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What does a recording of an agent show?**

In [ ]:
# 6.1 · An agent, as in Unit 5
from langfuse import get_client, propagate_attributes
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from agentic_ai.agents.tool_agent import ToolAgent
from agentic_ai.tools.calculator import CalculatorTool

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")
memory = InMemorySaver()

agent = ToolAgent(
    llm=llm,
    tools=[CalculatorTool()],
    checkpointer=memory
)

langfuse = get_client()

In [ ]:
# 6.2 · The agent, inside a span
with langfuse.start_as_current_observation(
    as_type="span",
    name="agent-query",
    input={"query": "Calculate (10 + 5) * 3 - 7"}
) as trace:

    with propagate_attributes(
        user_id="student_123",
        session_id="session_math_001",
        tags=["math", "calculator", "demo"],
        metadata={"agent_type": "ToolAgent", "model": "gemini-3.5-flash-lite"}
    ):
        result = agent.run(
            "Calculate (10 + 5) * 3 - 7",
            thread_id="trace_demo_001"
        )

        trace.update(output={"response": result['messages'][-1].text})

        print(f"Agent Response: {result['messages'][-1].text}")

langfuse.flush()
print("\n✅ Agent execution traced! Check your Langfuse dashboard.")

#### 6.3 · Let's look at the recording

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
agent-query

input:   Calculate (10 + 5) * 3 - 7
output:  the answer of the agent, with 38
labels:  user student_123, session session_math_001, tags math, calculator, demo
time:    27 s
inside:  nothing
```

This time includes two pauses of 4.5 seconds that only our test adds.

</details>

#### 6.4 · What do we take from this?

<details>
<summary>Click to show</summary>

**A wrapper records only the outside.**

---

**One sentence for today:** You cannot fix what you cannot see, and a trace shows only what you connect to it.

</details>

In [ ]:
# 6.5 · The handler, and a small chain
from langfuse.langchain import CallbackHandler
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

langfuse_handler = CallbackHandler()

llm = init_chat_model("gemini-3.5-flash-lite", model_provider="google_genai")
prompt = ChatPromptTemplate.from_template("What is {expression}? Show your work.")
chain = prompt | llm

In [ ]:
# 6.6 · Run the chain, with the handler
response = chain.invoke(
    {"expression": "15 * 7"},
    config={"callbacks": [langfuse_handler]}
)

print(f"Response: {response.text}")

langfuse = get_client()
langfuse.flush()
print("\n✅ LangChain chain traced via callback handler!")

#### 6.7 · Let's look at the recording

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
RunnableSequence
├─ ChatPromptTemplate
└─ ChatGoogleGenerativeAI   14 input tokens, 132 output tokens
```

</details>

In [ ]:
# 6.8 · Our agent, with the handler
from langchain_core.messages import HumanMessage

result = agent.graph.invoke(
    {"messages": [HumanMessage(content="Calculate (10 + 5) * 3 - 7")]},
    config={"configurable": {"thread_id": "trace_demo_002"},
            "callbacks": [langfuse_handler]},
)
print(result["messages"][-1].text)
langfuse.flush()

#### 6.9 · Let's look at the recording

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
LangGraph
├─ agent
│  ├─ ChatGoogleGenerativeAI   14 s, 160 input tokens
│  └─ should_continue          0 s
├─ tools
│  └─ calculator               0 s, input (10 + 5) * 3 - 7, result 38
└─ agent
   ├─ ChatGoogleGenerativeAI   10 s, 197 input tokens
   └─ should_continue          0 s
```

The model times include a pause of 4.5 seconds that only our test adds.

</details>

#### 6.10 · The two model calls

<details>
<summary>Click to show</summary>

**The model remembers nothing, so we must give it everything, every time.**

The second call sends the question again, plus the model's note asking for the calculator, plus the result.

</details>

<div style="height:44px"></div>

## Part 7 of 10 · Finding the right recording: labels

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**With a thousand conversations, how do we find the right one?**

In [ ]:
# 7.1 · A conversation: the first turn
from langfuse import get_client, propagate_attributes

langfuse = get_client()

session_id = "conversation_math_001"

with langfuse.start_as_current_observation(
    as_type="span",
    name="user-question-1",
    input={"question": "What is 5 + 3?"}
) as turn1:
    with propagate_attributes(session_id=session_id):
        answer1 = "5 + 3 = 8"
        turn1.update(output={"answer": answer1})
        print(f"Q1: What is 5 + 3?")
        print(f"A1: {answer1}")

langfuse.flush()

In [ ]:
# 7.2 · A conversation: the second turn
with langfuse.start_as_current_observation(
    as_type="span",
    name="user-question-2",
    input={"question": "What about 8 * 2?"}
) as turn2:
    with propagate_attributes(session_id=session_id):
        answer2 = "8 * 2 = 16"
        turn2.update(output={"answer": answer2})
        print(f"\nQ2: What about 8 * 2?")
        print(f"A2: {answer2}")

langfuse.flush()
print(f"\n✅ Both turns traced in session: {session_id}")
print("   Check Langfuse UI to see them grouped together!")

#### 7.3 · Let's look at the session

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
Session conversation_math_001

user-question-1
user-question-2
... and the same two again, from each earlier run
```

</details>

In [ ]:
# 7.4 · Three requests, two users
from langfuse import get_client, propagate_attributes

langfuse = get_client()

users = ["alice", "bob", "alice"]

for i, user_id in enumerate(users, 1):
    with langfuse.start_as_current_observation(
        as_type="span",
        name=f"user-request-{i}",
        input={"request": f"Request {i} from {user_id}"}
    ) as trace:
        with propagate_attributes(
            user_id=user_id,
            metadata={"request_number": i}
        ):
            trace.update(output={"processed": True, "user": user_id})
            print(f"Processed request {i} from user: {user_id}")

langfuse.flush()
print("\n✅ Traces tagged with user IDs!")
print("   In Langfuse UI, you can filter by user and see per-user analytics.")

#### 7.5 · All the labels

| Label | Type | What for | Example |
|:--|:--|:--|:--|
| `user_id` | text | who asked | `"user_123"` |
| `session_id` | text | which conversation | `"session_abc"` |
| `tags` | list of texts | short labels | `["production", "critical"]` |
| `metadata` | dictionary | any extra information | `{"env": "prod"}` |
| `version` | text | which version of your code | `"v2.1.0"` |

In [ ]:
# 7.6 · All labels at once
from langfuse import get_client, propagate_attributes

langfuse = get_client()

with langfuse.start_as_current_observation(
    as_type="span", name="complete-example", input={"action": "process-order"}
) as root:

    with propagate_attributes(
        user_id="customer_456",
        session_id="checkout_session_789",
        tags=["e-commerce", "checkout", "production"],
        metadata={"order_id": "ORD-12345", "total_amount": 299.99,
                  "payment_method": "credit_card", "shipping_address": "123 Main St"},
        version="1.2.3"
    ):
        with langfuse.start_as_current_observation(
            as_type="generation", name="validate-payment", model="gemini-3.5-flash-lite"
        ) as validation:
            validation.update(output={"valid": True})

        with langfuse.start_as_current_observation(
            as_type="span", name="send-confirmation"
        ) as confirmation:
            confirmation.update(output={"sent": True})

    root.update(output={"order_processed": True})

langfuse.flush()
print("✅ Complete trace with all attributes sent!")

<div style="height:44px"></div>

## Part 8 of 10 · Reading traces, and good habits

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**How do we read a trace, and which habits help?**

#### 8.1 · The words

**Trace:** the whole tree of one run.

**Observation:** one step in that tree. It has a type:

- **span:** normal work
- **generation:** a model call
- **event:** one moment, with no duration

#### 8.2 · Do not log sensitive data

No passwords and no keys.

No personal data, such as names, addresses or health problems.

You can give Langfuse a small function that hides such data before it is sent. That is called **masking**. It is not on by itself.

In [ ]:
# 8.3 · Environments
from langfuse import Langfuse

langfuse = Langfuse(
    environment="development"
)

print("✅ Langfuse configured for environment:",
      langfuse.environment if hasattr(langfuse, 'environment') else "default")

<div style="height:44px"></div>

## Part 9 of 10 · Your exercises

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What do you do at home?**

#### 9.1 · Exercise 1: a function

A function `process_numbers(numbers)` returns the sum, the average and the maximum, as a dictionary.

Add `@observe()`, call it, flush, and look for it in Langfuse.

In [ ]:
# 9.2 · Exercise 1, together
from langfuse import observe, get_client

@observe()
def process_numbers(numbers: list):
    return {
        "sum": sum(numbers),
        "average": sum(numbers) / len(numbers),
        "maximum": max(numbers),
    }

result = process_numbers([10, 20, 30, 40, 50])
print(result)
langfuse = get_client()
langfuse.flush()

#### 9.3 · Let's look at the recording

<details>
<summary>Click to show</summary>

**In our test run on 4 November, Langfuse held this:**

```
process_numbers

input:   args, with our list
output:  sum 150, average 30.0, maximum 50
```

</details>

#### 9.4 · Exercise 2: an agent with labels

A `ToolAgent` with `CalculatorTool`, inside a span, with `user_id`, `session_id`, `tags` and `metadata`.

The question: “What is (25 * 4) + (100 / 5)?”

#### 9.5 · Exercise 3: two questions, one session

First: “What is 10 + 5?” Then: “What is that result multiplied by 3?”

`session_id` is only a label for Langfuse.

`thread_id` gives the agent its memory.

#### 9.6 · Exercise 4: steps inside steps

```
process-order
├─ validate-order
└─ generate-confirmation
   └─ send-email
```

All details are in your own notebook.

<div style="height:44px"></div>

## Part 10 of 10 · Wrap-up

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span></div>

**What should you remember from today?**

#### 10.1 · Five things

**1.** A trace is a flight recorder for your agent: every step that you connect, with what went in and what came out.

**2.** `@observe` records a function, and `with` records steps inside steps.

**3.** A wrapper only records the outside. A callback handler connects the inside.

**4.** Sessions, users, tags and metadata help you find the right recording.

**5.** Always flush, and do not send secrets or personal data.

---

**You cannot fix what you cannot see, and a trace shows only what you connect to it.**

**At home:** the exercises, up to Exercise 3.

**Next time, Unit 8:** we give our agents knowledge from our own documents.